## Imports and Config

In [ ]:
%load_ext autoreload
%autoreload 2

Custom imports from the `config/settings.py` file.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

In [ ]:
from haystack import Pipeline, component
from hayhooks import BasePipelineWrapper
from haystack_integrations.components.retrievers.qdrant import QdrantEmbeddingRetriever

from openai import OpenAI
from openai.types.create_embedding_response import CreateEmbeddingResponse

In [ ]:
from sem_rag.config.settings import VLLM, SEMDataset, Qdrant, Evaluation
from sem_rag.storage.qdrant import get_document_store
from sem_rag.embedding.embedder import CLIPEmbedder

In [ ]:
INSTRUCTION = "Represent the user's input."

TOP_K = 10

In [ ]:
# Lista di ID delle immagini
QUERIES = [
    "38",
]

In [ ]:
client = OpenAI(base_url=VLLM.API_URL, api_key=VLLM.API_KEY)

In [ ]:
qwen_qdrant = get_document_store(Qdrant.COLLECTION_QWEN, Qdrant.EMBEDDING_DIM_QWEN)
clip_qdrant = get_document_store(Qdrant.COLLECTION_CLIP, Qdrant.EMBEDDING_DIM_CLIP)

In [ ]:
def get_crop_path(crops_dir: str | Path, image_id: str) -> Path:
    crops_dir = Path(crops_dir)

    matches = list(crops_dir.glob(f"{image_id}_*.npy"))

    if not matches:
        raise FileNotFoundError(f"No crop found for image ID: {image_id}")

    if len(matches) > 1:
        raise ValueError(f"Multiple crops found for image ID {image_id}: {matches}")

    return matches[0]

# Gabor

In [ ]:
import numpy as np

from sem_rag.retrieval.gabor import GaborRetriever


retriever = GaborRetriever(Evaluation.GABOR_FILE)

query_image = np.load(get_crop_path(SEMDataset.CROPS, QUERIES[0]))

results = retriever.retrieve(
    query_image,
    top_k=5,
)

print("Results:", results)

# CLIP

In [ ]:
clip_embedder = CLIPEmbedder()
retriever = QdrantEmbeddingRetriever(clip_qdrant)

In [ ]:
# Generate the CLIP text embedding
query_embedding = clip_embedder.embed_image(get_crop_path(SEMDataset.CROPS, QUERIES[0]))

In [ ]:
# Retrieve the most similar images
clip_results = retriever.run(
    query_embedding=query_embedding,
    top_k=TOP_K,
).get("documents", [])

In [ ]:
for result in clip_results:
    print(result)

# RAG System

In [ ]:
@component
class QueryEmbedder():
    def __init__(self, model: str, api_base_url: str, api_key: str):
        self.model = model
        self.client = OpenAI(base_url=api_base_url, api_key=api_key)

    @component.output_types(embedding=list[float])
    def run(self, text: str = "", image_data_urls: list[str] | None = None):
        if image_data_urls:
            messages = [
                {"role": "system", "content": [{"type": "text", "text": INSTRUCTION,}]},
                {"role": "user", "content": [
                        {"type": "image_url", "image_url": {"url": image_data_urls[0]}},
                        {"type": "text", "text": ""},
                ]},
                {"role": "assistant", "content": [{"type": "text", "text": ""}]},
            ]
        else:
            messages = [
                {"role": "system", "content": [{"type": "text", "text": INSTRUCTION,}]},
                {"role": "user", "content": [{"type": "text", "text": text}]},
                {"role": "assistant", "content": [{"type": "text", "text": ""}]},
            ]

        response: CreateEmbeddingResponse = self.client.post(
            "/embeddings",
            cast_to=CreateEmbeddingResponse,
            body={
                "messages": messages,
                "model": self.model,
                "encoding_format": "float",
                "continue_final_message": True,
                "add_special_tokens": True,
            },
        )

        return {"embedding": response.data[0].embedding}

In [ ]:
class PipelineWrapper(BasePipelineWrapper):
    def setup(self) -> None:
        document_store = get_document_store(Qdrant.COLLECTION_QWEN, Qdrant.EMBEDDING_DIM_QWEN)

        self.retrieval_pipeline = Pipeline()

        self.retrieval_pipeline.add_component("query_embedder", QueryEmbedder(model=VLLM.EMBEDDING_MODEL, api_base_url=VLLM.API_URL, api_key=VLLM.API_KEY))
        self.retrieval_pipeline.add_component("retriever", QdrantEmbeddingRetriever(document_store=document_store, top_k=TOP_K))

        self.retrieval_pipeline.connect("query_embedder.embedding", "retriever")

    def retrieve_documents(self, question: str, images: list[str]):
        retrieval_run_args = {
            "query_embedder": {
                "text": question,
                "image_data_urls": images,
            },
        }

        response = self.retrieval_pipeline.run(retrieval_run_args)

        documents = (response.get("retriever", {}).get("documents", []))

        return documents

In [ ]:
qwen_retriever = PipelineWrapper()
qwen_retriever.setup()

In [ ]:
qwen_results = qwen_retriever.retrieve_documents(
    QUERIES[0],
    [],
)

for result in qwen_results:
    print(result.id, result.content)

# Results